# 03 — Evaluation and figures

Computes every metric reported in Chapter 3 and produces the figures. No GPU needed: it starts
from `resultados_finales.json`, written by the consolidation cell of notebook 01.

Requires `skg.py`. Writes `evaluacion.py`.

## The evaluation module

Two decisions worth flagging. The directional metric only counts scenarios whose annotation
carries a valid severity level, since otherwise any output would count as a hit. And McNemar
is used in its exact form, which is the right choice when the number of discordant pairs is
small.

In [ ]:
%%writefile ../src/evaluacion.py
"""Offline analysis of the results. No GPU needed.

Runs locally over the JSON files:
    python evaluacion.py results1.json results2.json ...

Produces:
  - Exact and directional accuracy per agent class
  - Parse failure rates and ambiguity types
  - McNemar test over the discordant pairs
  - Confusion matrix
  - Validator outcomes (hallucination, ego-frame drift)
  - Captioning metrics (BLEU-4, METEOR, ROUGE-L)
"""
import numpy as np
import sys
import json
from statsmodels.stats.contingency_tables import mcnemar
from collections import defaultdict, Counter
from skg import normalizar_accion, NIVEL, CLASE_DE


# ===========================================================================
# 1. LOADING
# ===========================================================================
def cargar(*rutas):
    """Concatenate several result files. Warns if one is missing."""
    todos = []
    for ruta in rutas:
        try:
            with open(ruta) as f:
                datos = json.load(f)
            print(f"  {ruta}: {len(datos)} escenarios")
            todos.extend(datos)
        except FileNotFoundError:
            print(f"  [missing] {ruta}")
    return todos


# ===========================================================================
# 2. ACTION METRICS
# ===========================================================================
def evaluar(resultados):
    """Compute the metrics per agent class.

    Returns (stats, detalle):
      stats   -> counters per class
      detalle -> one row per valid scenario, for McNemar and the confusion matrix
    """
    stats = defaultdict(lambda: defaultdict(int))
    detalle = []

    for r in resultados:
        tipo = r["tipo"]
        gt, _ = normalizar_accion(r["sugerencia_gt"])
        con, tipo_con = normalizar_accion(r["con_skg"]["accion_cruda"])
        sin, tipo_sin = normalizar_accion(r["sin_skg"]["accion_cruda"])

        stats[tipo]["n"] += 1

        # Output quality. These are first-class metrics, not noise.
        if r["con_skg"].get("fallo"):
            stats[tipo]["fallos_con"] += 1
        if r["sin_skg"].get("fallo"):
            stats[tipo]["fallos_sin"] += 1
        if tipo_con == "condicional":
            stats[tipo]["condicional_con"] += 1
        if tipo_con == "secuencia":
            stats[tipo]["secuencia_con"] += 1
        if tipo_sin == "condicional":
            stats[tipo]["condicional_sin"] += 1
        if tipo_sin == "secuencia":
            stats[tipo]["secuencia_sin"] += 1

        # If the ground truth carries no recognisable action, skip the match
        if gt is None:
            stats[tipo]["gt_sin_clave"] += 1
            continue

        con_ok = (con == gt)
        sin_ok = (sin == gt)
        stats[tipo]["exacto_con"] += int(con_ok)
        stats[tipo]["exacto_sin"] += int(sin_ok)

        # Directional: hit the label or overshoot it. Only when the GT has a valid
        # level; otherwise any output would count as a hit.
        nivel_gt = NIVEL.get(gt, 0)
        if nivel_gt > 0:
            stats[tipo]["dir_n"] += 1
            stats[tipo]["dir_con"] += int(con is not None and NIVEL.get(con, 0) >= nivel_gt)
            stats[tipo]["dir_sin"] += int(sin is not None and NIVEL.get(sin, 0) >= nivel_gt)

        detalle.append({
            "indice": r["indice"], "tipo": tipo, "gt": gt,
            "con": con, "sin": sin, "con_ok": con_ok, "sin_ok": sin_ok,
            "hazard": r["con_skg"].get("hazard"),
        })

    return stats, detalle


def tabla_acciones(stats):
    """Print the main results table."""
    print(f"\n{'Clase':<16}{'n':>4} | {'Ex CON':>7}{'Ex SIN':>7} | "
          f"{'Dir CON':>8}{'Dir SIN':>8} | {'Cond':>5}{'Sec':>5}{'Fallo':>6}")
    print("-" * 72)

    total = defaultdict(int)
    for clase, s in stats.items():
        print(f"{clase:<16}{s['n']:>4} | {s['exacto_con']:>7}{s['exacto_sin']:>7} | "
              f"{s['dir_con']:>8}{s['dir_sin']:>8} | "
              f"{s['condicional_con']:>5}{s['secuencia_con']:>5}"
              f"{s['fallos_con'] + s['fallos_sin']:>6}")
        for k, v in s.items():
            total[k] += v

    print("-" * 72)
    print(f"{'TOTAL':<16}{total['n']:>4} | {total['exacto_con']:>7}{total['exacto_sin']:>7} | "
          f"{total['dir_con']:>8}{total['dir_sin']:>8} | "
          f"{total['condicional_con']:>5}{total['secuencia_con']:>5}"
          f"{total['fallos_con'] + total['fallos_sin']:>6}")

    n = total["n"] or 1
    dn = total["dir_n"] or 1
    print(f"\n  Exacta:      CON {100*total['exacto_con']/n:.1f}%  "
          f"SIN {100*total['exacto_sin']/n:.1f}%")
    print(f"  Direccional: CON {100*total['dir_con']/dn:.1f}%  "
          f"SIN {100*total['dir_sin']/dn:.1f}%  (sobre {total['dir_n']} casos validos)")
    print(f"  GT sin accion reconocible (excluidos): {total['gt_sin_clave']}")
    print(f"  Condicionales sin resolver: CON {total['condicional_con']}  "
          f"SIN {total['condicional_sin']}")
    print(f"  Secuencias compuestas:      CON {total['secuencia_con']}  "
          f"SIN {total['secuencia_sin']}")
    return total


# ===========================================================================
# 3. McNEMAR (paired design)
# ===========================================================================
def mcnemar_test(detalle, etiqueta=""):
    """McNemar test over the discordant pairs.

    Only the cells where the two conditions differ say anything about which is
    better; the agreements carry no information.
    """
    a = sum(1 for d in detalle if d["con_ok"] and d["sin_ok"])          # both right
    b = sum(1 for d in detalle if d["con_ok"] and not d["sin_ok"])      # only WITH
    c = sum(1 for d in detalle if not d["con_ok"] and d["sin_ok"])      # only WITHOUT
    e = sum(1 for d in detalle if not d["con_ok"] and not d["sin_ok"])  # both wrong

    print(f"\n--- McNemar {etiqueta} (n={len(detalle)}) ---")
    print(f"  both right: {a:<4} only WITH: {b}")
    print(f"  only WITHOUT: {c:<4} both wrong: {e}")

    # exact=True gives the binomial test, which is right when b+c is small
    res = mcnemar(np.array([[a, b], [c, e]]), exact=True)
    print(f"  p = {res.pvalue:.4f} -> "
          f"{'significant' if res.pvalue < 0.05 else 'not significant'} (alpha=0.05)")
    return res.pvalue


# ===========================================================================
# 4. FAILURE DIAGNOSIS
# ===========================================================================
def matriz_confusion(detalle, solo_ambos_fallan=False, top=12):
    """Show the most frequent (ground truth -> predicted) pairs."""
    filas = [d for d in detalle if not (d["con_ok"] or d["sin_ok"])] if solo_ambos_fallan else detalle
    titulo = "both wrong" if solo_ambos_fallan else "all cases"
    print(f"\n--- Confusion CON SKG ({titulo}, {len(filas)} casos) ---")
    for (gt, pred), cuenta in Counter((d["gt"], d["con"]) for d in filas).most_common(top):
        marca = "ok" if gt == pred else "  "
        print(f"  {marca} {str(gt):<24} -> {str(pred):<24} ({cuenta})")


def precision_router(detalle):
    """Hazard agent accuracy: class of the chosen hazard vs the annotated class."""
    router = defaultdict(lambda: {"ok": 0, "n": 0})
    for d in detalle:
        clase_elegida = CLASE_DE.get(d.get("hazard"), "?")
        router[d["tipo"]]["n"] += 1
        router[d["tipo"]]["ok"] += int(clase_elegida == d["tipo"])

    print("\n--- Router accuracy (class of the chosen hazard) ---")
    for clase, s in router.items():
        pct = 100 * s["ok"] / s["n"] if s["n"] else 0
        print(f"  {clase:<16} {s['ok']:>3}/{s['n']:<3} ({pct:.0f}%)")


def distribucion_acciones(resultados):
    """Compare the action distribution of GT, WITH and WITHOUT.

    Shows the commitment collapse: without the graph the system falls back on
    non-committal actions.
    """
    gt = Counter(normalizar_accion(r["sugerencia_gt"])[0] for r in resultados)
    con = Counter(normalizar_accion(r["con_skg"]["accion_cruda"])[0] for r in resultados)
    sin = Counter(normalizar_accion(r["sin_skg"]["accion_cruda"])[0] for r in resultados)

    print("\n--- Action distribution ---")
    print(f"  {'Action':<26}{'GT':>5}{'WITH':>6}{'W/O':>6}")
    for accion in sorted(set(gt) | set(con) | set(sin), key=lambda a: -NIVEL.get(a, 0)):
        print(f"  {str(accion):<26}{gt.get(accion,0):>5}{con.get(accion,0):>6}{sin.get(accion,0):>6}")


# ===========================================================================
# 5. VALIDATOR
# ===========================================================================
def resultados_validador(resultados):
    """Pass rate, hallucination rate and ego-frame drift."""
    total = aprobados = ego_ok = con_alucinacion = 0
    for r in resultados:
        v = r["con_skg"].get("validacion")
        if not v:
            continue
        total += 1
        aprobados += int(v.get("verdict") == "pass")
        ego_ok += int(v.get("ego_frame_ok") is True)
        con_alucinacion += int(bool(v.get("hallucinated_claims")))

    if not total:
        print("\n--- Validator: no verdicts ---")
        return

    print(f"\n--- Validator ({total} cases) ---")
    print(f"  Passed:           {aprobados}/{total} ({100*aprobados/total:.0f}%)")
    print(f"  With hallucination: {con_alucinacion}/{total} ({100*con_alucinacion/total:.0f}%)")
    print(f"  Ego-frame drift:  {total-ego_ok}/{total} ({100*(total-ego_ok)/total:.0f}%)")


# ===========================================================================
# 6. CAPTIONING
# ===========================================================================
def metricas_caption(resultados):
    """BLEU-4, METEOR and ROUGE-L of the generated captions against DRAMA's."""
    try:
        import nltk
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
        from nltk.translate.meteor_score import meteor_score
        from rouge_score import rouge_scorer
    except ImportError:
        print("\n[captioning: pip install nltk rouge-score]")
        return

    suavizado = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    bleu = meteor = rouge = 0.0
    n = 0
    for r in resultados:
        hip = r["con_skg"].get("caption", "")
        ref = r.get("caption_gt", "")
        if not hip or not ref:
            continue
        n += 1
        hip_tok, ref_tok = hip.split(), ref.split()
        bleu += sentence_bleu([ref_tok], hip_tok, weights=(0.25,) * 4,
                              smoothing_function=suavizado)
        meteor += meteor_score([ref_tok], hip_tok)
        rouge += scorer.score(ref, hip)["rougeL"].fmeasure

    if not n:
        print("\n--- Captioning: no captions generated ---")
        return

    print(f"\n--- Captioning ({n} captions) ---")
    print(f"  BLEU-4:  {bleu/n:.4f}")
    print(f"  METEOR:  {meteor/n:.4f}")
    print(f"  ROUGE-L: {rouge/n:.4f}")
    print("  (CIDEr needs pycocoevalcap, optional)")


# ===========================================================================
# 7. SINGLE-FRAME vs TEMPORAL
# ===========================================================================
def comparar_temporal(base, temporal, clase=None):
    """Compare the same sample under single-frame and multi-frame perception."""
    idx_temporal = {r["indice"]: r for r in temporal}
    pares = [(b, idx_temporal[b["indice"]]) for b in base
             if b["indice"] in idx_temporal and (clase is None or b["tipo"] == clase)]

    if not pares:
        print("\n--- Temporal: no comparable pairs ---")
        return

    sf = tf = 0
    for b, t in pares:
        gt, _ = normalizar_accion(b["sugerencia_gt"])
        a_sf, _ = normalizar_accion(b["con_skg"]["accion_cruda"])
        a_tf, _ = normalizar_accion(t["con_skg"]["accion_cruda"])
        sf += int(a_sf == gt)
        tf += int(a_tf == gt)

    etiqueta = clase or "all classes"
    print(f"\n--- Single-frame vs temporal ({etiqueta}, {len(pares)} pares) ---")
    print(f"  Exacta 1 frame:  {sf}/{len(pares)}")
    print(f"  Exacta temporal: {tf}/{len(pares)}")

    
def tests(datos, etiqueta):
    """Exact and directional match for one condition against its baseline.

    Prints both metrics with their McNemar p-value in a single block.
    """
    ec, es, dc, ds = [], [], [], []
    for r in datos:
        gt, _ = normalizar_accion(r["sugerencia_gt"])
        if not gt: continue
        c, _ = normalizar_accion(r["con_skg"]["accion_cruda"])
        s, _ = normalizar_accion(r["sin_skg"]["accion_cruda"])
        ec.append(c == gt); es.append(s == gt)
        if NIVEL.get(gt, 0) > 0:
            dc.append(c is not None and NIVEL.get(c,0) >= NIVEL[gt])
            ds.append(s is not None and NIVEL.get(s,0) >= NIVEL[gt])

    def mc(x, y, nombre):
        a=sum(1 for p,q in zip(x,y) if p and q);     b=sum(1 for p,q in zip(x,y) if p and not q)
        c_=sum(1 for p,q in zip(x,y) if not p and q); e=sum(1 for p,q in zip(x,y) if not p and not q)
        pv = mcnemar(np.array([[a,b],[c_,e]]), exact=True).pvalue
        print(f"  {nombre:<13} CON {sum(x):>3}/{len(x)} ({100*sum(x)/len(x):>4.1f}%)  "
              f"SIN {sum(y):>3}/{len(y)} ({100*sum(y)/len(y):>4.1f}%)  |  "
              f"soloCON={b:<3} soloSIN={c_:<3} ambos✗={e:<3} p={pv:.4f}{'  ***' if pv<0.05 else ''}")

    print(f"\n=== {etiqueta} (n={len(ec)}) ===")
    mc(ec, es, "Exacta"); mc(dc, ds, "Direccional")

# ===========================================================================
# 8. FULL REPORT
# ===========================================================================
def informe(resultados, temporal=None):
    """Run the whole analysis in one pass."""
    stats, detalle = evaluar(resultados)
    tabla_acciones(stats)
    mcnemar_test(detalle, "(CON vs SIN)")
    matriz_confusion(detalle, solo_ambos_fallan=True)
    precision_router(detalle)
    distribucion_acciones(resultados)
    resultados_validador(resultados)
    metricas_caption(resultados)
    if temporal:
        comparar_temporal(resultados, temporal, clase="Vehicle")
        comparar_temporal(resultados, temporal)
    return stats, detalle


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print(__doc__)
        sys.exit(1)
    print("Loading results:")
    datos = cargar(*sys.argv[1:])
    print(f"Total: {len(datos)} escenarios")
    informe(datos)

In [ ]:
import sys, os, json, re
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import mcnemar

sys.path.insert(0, "../src")
from skg import SKG, CLASE_DE, NIVEL, normalizar_accion
from evaluacion import (evaluar, tabla_acciones, mcnemar_test, matriz_confusion,
                        precision_router, distribucion_acciones, resultados_validador)

RESULTADOS = os.environ.get("RESULTS_DIR", "../results")
FIGURAS = "figuras"
os.makedirs(FIGURAS, exist_ok=True)

TIPOS = ["Vehicle", "Pedestrian", "Cyclist", "Infrastructure"]

datos = json.load(open(f"{RESULTADOS}/resultados_finales.json"))
print(f"Scenarios: {len(datos)}", dict(Counter(r["tipo"] for r in datos)))

### Figure style

Colours differ in lightness as well as hue, so the figures still read if the document is
printed in black and white.

In [ ]:
COLOR_A   = "#2F4B7C"   # with the knowledge graph
COLOR_B   = "#A8B8D0"   # reasoning without knowledge
COLOR_C   = "#7D8C9E"   # direct inference baseline
COLOR_GT  = "#4D4D4D"   # ground truth
COLOR_ALT = "#C1666B"   # highlight

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "figure.dpi": 110,
})


def guardar(fig, nombre):
    """PDF for the document, PNG for a quick look."""
    for ext in ("pdf", "png"):
        fig.savefig(f"{FIGURAS}/{nombre}.{ext}", bbox_inches="tight", dpi=300)
    print(f"  saved: {FIGURAS}/{nombre}.pdf")

## 3.1 — Contribution of the normative knowledge

Three conditions over the same scenarios. A uses the full graph; B keeps the reasoning stages
but has no catalogue and no subgraph; C derives the action straight from the perception.

Contrasting A with B isolates the contribution of the knowledge. Contrasting B with C isolates
that of the reasoning scaffolding.

In [ ]:
def recoger(metrica):
    """Per-scenario hit vectors for the three conditions, aligned by index."""
    a, b, c = [], [], []
    for r in datos:
        gt, _ = normalizar_accion(r["sugerencia_gt"])
        if not gt or not r.get("B_sin_conocimiento"):
            continue
        if metrica == "direccional" and NIVEL.get(gt, 0) == 0:
            continue
        va, _ = normalizar_accion(r["A_con_skg"]["accion_cruda"])
        vb, _ = normalizar_accion(r["B_sin_conocimiento"]["accion_cruda"])
        vc, _ = normalizar_accion(r["C_baseline"]["accion_cruda"])
        if metrica == "exacta":
            a.append(va == gt); b.append(vb == gt); c.append(vc == gt)
        else:
            supera = lambda v: v is not None and NIVEL.get(v, 0) >= NIVEL[gt]
            a.append(supera(va)); b.append(supera(vb)); c.append(supera(vc))
    return a, b, c


def mc(x, y, etiqueta):
    """McNemar over the discordant pairs of two conditions."""
    aa = sum(1 for p, q in zip(x, y) if p and q)
    bb = sum(1 for p, q in zip(x, y) if p and not q)
    cc = sum(1 for p, q in zip(x, y) if not p and q)
    ee = sum(1 for p, q in zip(x, y) if not p and not q)
    pv = mcnemar(np.array([[aa, bb], [cc, ee]]), exact=True).pvalue
    print(f"  {etiqueta:<38} {sum(x):>3} vs {sum(y):>3} | "
          f"only_1={bb:<3} only_2={cc:<3} p={pv:.4f}{'  ***' if pv < 0.05 else ''}")
    return aa, bb, cc, ee, pv


for metrica in ["exacta", "direccional"]:
    a, b, cc = recoger(metrica)
    n = len(a)
    print(f"\n=== {metrica.upper()} (n={n}) ===")
    print(f"  A) full SKG:         {sum(a):>3} ({100*sum(a)/n:.1f}%)")
    print(f"  B) reasoning only:   {sum(b):>3} ({100*sum(b)/n:.1f}%)")
    print(f"  C) direct inference: {sum(cc):>3} ({100*sum(cc)/n:.1f}%)")
    mc(a, b,  "A vs B (contribution of the knowledge)")
    mc(b, cc, "B vs C (contribution of the scaffolding)")
    mc(a, cc, "A vs C (total effect)")

### Which condition solves what

How many scenarios each condition gets right on its own. This isolates the contribution better
than the aggregate percentage does.

In [ ]:
combos = Counter()
for r in datos:
    gt, _ = normalizar_accion(r["sugerencia_gt"])
    if not gt or not r.get("B_sin_conocimiento"):
        continue
    a = normalizar_accion(r["A_con_skg"]["accion_cruda"])[0] == gt
    b = normalizar_accion(r["B_sin_conocimiento"]["accion_cruda"])[0] == gt
    c = normalizar_accion(r["C_baseline"]["accion_cruda"])[0] == gt
    combos[(a, b, c)] += 1

n = sum(combos.values())
print(f"n = {n}   (A, B, C):")
for k, v in sorted(combos.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:>4} ({100*v/n:.1f}%)")
print(f"\nOnly A right: {combos[(True,False,False)]} "
      f"({100*combos[(True,False,False)]/n:.1f}%)")
print(f"All three wrong: {combos[(False,False,False)]} "
      f"({100*combos[(False,False,False)]/n:.1f}%)")

### By agent class, and the reweighted estimate

The sample is balanced by class but DRAMA is not, so the overall mean does not represent
performance over the real distribution. The reweighting applies DRAMA's class proportions to
the per-class performance observed here.

In [ ]:
# DRAMA class proportions over the risk-positive scenarios
PESOS = {"Vehicle": 12273/17066, "Pedestrian": 2909/17066,
         "Cyclist": 435/17066, "Infrastructure": 1449/17066}

pc = defaultdict(lambda: defaultdict(int))
for r in datos:
    gt, _ = normalizar_accion(r["sugerencia_gt"])
    if not gt or not r.get("B_sin_conocimiento"):
        continue
    d = pc[r["tipo"]]
    va, _ = normalizar_accion(r["A_con_skg"]["accion_cruda"])
    vb, _ = normalizar_accion(r["B_sin_conocimiento"]["accion_cruda"])
    vc, _ = normalizar_accion(r["C_baseline"]["accion_cruda"])
    d["n"] += 1
    d["A"] += (va == gt); d["B"] += (vb == gt); d["C"] += (vc == gt)
    if NIVEL.get(gt, 0) > 0:
        d["dn"] += 1
        supera = lambda v: v is not None and NIVEL.get(v, 0) >= NIVEL[gt]
        d["dA"] += supera(va); d["dB"] += supera(vb); d["dC"] += supera(vc)

print(f"{'class':<16}{'n':>4}{'A':>7}{'B':>7}{'C':>7}  |{'dA':>7}{'dB':>7}{'dC':>7}")
for t in TIPOS:
    d = pc[t]
    print(f"{t:<16}{d['n']:>4}{100*d['A']/d['n']:>6.1f}%{100*d['B']/d['n']:>6.1f}%"
          f"{100*d['C']/d['n']:>6.1f}%  |{100*d['dA']/d['dn']:>6.1f}%"
          f"{100*d['dB']/d['dn']:>6.1f}%{100*d['dC']/d['dn']:>6.1f}%")

rep = lambda campo, base: sum(PESOS[t] * pc[t][campo] / pc[t][base] for t in PESOS)
print("\n--- Reweighted to the DRAMA distribution ---")
for k in "ABC":
    print(f"  {k}: exact {100*rep(k,'n'):.1f}%  directional {100*rep('d'+k,'dn'):.1f}%")

### Figure — exact and directional match by agent class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
x = np.arange(len(TIPOS))
ancho = 0.26

for ax, (base, titulo, prefijo) in zip(axes, [
        ("n", "Exact action match", ""),
        ("dn", "Directional match", "d")]):
    for i, (letra, color, etiqueta) in enumerate(zip(
            "ABC", [COLOR_A, COLOR_B, COLOR_C],
            ["A: full SKG", "B: reasoning only", "C: baseline"])):
        vals = [100 * pc[t][prefijo + letra] / pc[t][base] for t in TIPOS]
        ax.bar(x + (i - 1) * ancho, vals, ancho, label=etiqueta, color=color)
    ax.set_xticks(x); ax.set_xticklabels(TIPOS, fontsize=9)
    ax.set_ylabel("Scenarios (%)"); ax.set_title(titulo, fontsize=10)
    ax.set_ylim(0, 100)

axes[0].legend(frameon=False, fontsize=8.5)
plt.tight_layout()
guardar(fig, "fig_precision_por_clase")
plt.show()

### Figure — action distribution

This is where the commitment collapse shows: without normative knowledge the system retreats
to the least committal actions and abandons the specific manoeuvres.

In [ ]:
gt_d = Counter(normalizar_accion(r["sugerencia_gt"])[0] for r in datos)
a_d  = Counter(normalizar_accion(r["A_con_skg"]["accion_cruda"])[0] for r in datos)
b_d  = Counter(normalizar_accion(r["B_sin_conocimiento"]["accion_cruda"])[0]
               for r in datos if r.get("B_sin_conocimiento"))
c_d  = Counter(normalizar_accion(r["C_baseline"]["accion_cruda"])[0] for r in datos)

acciones = sorted([a for a in set(gt_d) | set(a_d) if a], key=lambda a: -NIVEL.get(a, 0))
x = np.arange(len(acciones))
ancho = 0.21

fig, ax = plt.subplots(figsize=(10, 4.2))
for i, (cuenta, color, etiqueta) in enumerate(zip(
        [gt_d, a_d, b_d, c_d], [COLOR_GT, COLOR_A, COLOR_B, COLOR_C],
        ["Ground truth", "A: full SKG", "B: reasoning only", "C: baseline"])):
    ax.bar(x + (i - 1.5) * ancho, [cuenta.get(a, 0) for a in acciones],
           ancho, label=etiqueta, color=color)

ax.set_xticks(x)
ax.set_xticklabels([a.replace(" ", "\n", 1) for a in acciones], fontsize=8.5)
ax.set_ylabel("Number of scenarios")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
guardar(fig, "fig_distribucion_acciones")
plt.show()

## 3.2 — Perception agent inputs

Three inputs over the same pipeline: a single frame, a ten-frame sequence, and a single frame
with the annotated bounding box.

In [ ]:
def comparar_entrada(campo, etiqueta):
    """Paired comparison of one perceptual variant against single-frame."""
    e1, e2, d1, d2 = [], [], [], []
    for r in datos:
        if not r.get(campo):
            continue
        gt, _ = normalizar_accion(r["sugerencia_gt"])
        if not gt:
            continue
        a, _ = normalizar_accion(r["A_con_skg"]["accion_cruda"])
        b, _ = normalizar_accion(r[campo]["accion_cruda"])
        e1.append(a == gt); e2.append(b == gt)
        if NIVEL.get(gt, 0) > 0:
            supera = lambda v: v is not None and NIVEL.get(v, 0) >= NIVEL[gt]
            d1.append(supera(a)); d2.append(supera(b))
    print(f"\n=== single-frame vs {etiqueta} (n={len(e1)}) ===")
    mc(e1, e2, "Exact")
    mc(d1, d2, "Directional")


comparar_entrada("temporal_10f", "10 frames")
comparar_entrada("oraculo_bbox", "bounding box")

### What each input changes in the perception

Each one improves a different aspect: the temporal sequence picks up the traffic-light state,
the box identifies the annotated object. Neither improves the decision.

In [ ]:
def perfil(campo, etiqueta):
    """Traffic-light detection, influence attribution and parse failures."""
    filas = [r[campo] for r in datos if r.get(campo)]
    percepciones = [f["percepcion"] for f in filas if f.get("percepcion")]

    sem = Counter(p.get("traffic_light_state") for p in percepciones)
    definido = sum(v for k, v in sem.items() if k in ("red", "amber", "green"))
    infl = Counter(o.get("influence_on_ego_path")
                   for p in percepciones for o in p.get("objects", []))
    tot = sum(infl.values()) or 1
    fallos = sum(1 for f in filas if f.get("fallo"))

    print(f"{etiqueta:<16} traffic light {definido:>4}/{len(percepciones)} | "
          f"direct {infl.get('direct',0):>4} ({100*infl.get('direct',0)/tot:>4.1f}%) "
          f"potential {infl.get('potential',0):>4} none {infl.get('none',0):>4} | "
          f"failures {fallos:>2}")
    return 100*definido/len(percepciones)


sem_1f  = perfil("A_con_skg", "1 frame")
sem_10f = perfil("temporal_10f", "10 frames")
sem_bb  = perfil("oraculo_bbox", "bounding box")

### Figure — perception improves, the decision does not

In [ ]:
def exactitud(campo):
    n = ex = 0
    for r in datos:
        if not r.get(campo):
            continue
        gt, _ = normalizar_accion(r["sugerencia_gt"])
        if not gt:
            continue
        n += 1
        ex += normalizar_accion(r[campo]["accion_cruda"])[0] == gt
    return 100*ex/n

etiquetas = ["1 frame", "10 frames", "Bounding box"]
semaforo  = [sem_1f, sem_10f, sem_bb]
accion    = [exactitud(c) for c in ["A_con_skg", "temporal_10f", "oraculo_bbox"]]

fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(3); w = 0.35
ax.bar(x - w/2, semaforo, w, color=COLOR_ALT, label="Traffic-light detected")
ax.bar(x + w/2, accion,   w, color=COLOR_A,   label="Exact action match")
for i, (s, a) in enumerate(zip(semaforo, accion)):
    ax.text(i - w/2, s + 1, f"{s:.0f}%", ha="center", fontsize=10)
    ax.text(i + w/2, a + 1, f"{a:.0f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(etiquetas)
ax.set_ylabel("Percentage of scenarios"); ax.set_ylim(0, 60)
ax.legend(frameon=False)
plt.tight_layout()
guardar(fig, "fig_disociacion")
plt.show()

## 3.3 — Where the pipeline fails

Performance is broken down by stage. Perception counts as correct when it mentions an object
of the annotated class; routing, when the chosen hazard belongs to that class. The failure
rate conditioned on both working is what isolates the final stage.

In [ ]:
# Keywords used to decide whether the perception named an object of the annotated class
TERMINOS = {
    "Vehicle":        ["car", "van", "truck", "bus", "vehicle", "sedan", "motorcycle", "taxi", "suv"],
    "Pedestrian":     ["pedestrian", "person", "man", "woman", "child", "people", "worker", "guard"],
    "Cyclist":        ["cyclist", "bicycle", "bike", "biker"],
    "Infrastructure": ["cone", "light", "sign", "barrier", "construction", "pole", "signal", "crossing"],
}


def menciona(texto, clase):
    return any(re.search(rf"\b{w}", texto.lower()) for w in TERMINOS[clase])


def etapas(r, campo="A_con_skg"):
    """(saw the object, routed to the right class, got the action right)."""
    f = r.get(campo)
    if not f:
        return None
    p = f.get("percepcion")
    gt, _ = normalizar_accion(r["sugerencia_gt"])
    if not p or not gt:
        return None
    objs = p.get("objects", [])
    return (menciona(" ".join(str(o.get("label", "")) for o in objs), r["tipo"]),
            CLASE_DE.get(f.get("hazard")) == r["tipo"],
            normalizar_accion(f.get("accion_cruda", ""))[0] == gt)


fallo_final = {}
for campo, etiqueta in [("A_con_skg", "1 frame"), ("temporal_10f", "10 frames"),
                        ("oraculo_bbox", "bounding box")]:
    combos = Counter(e for r in datos if (e := etapas(r, campo)))
    n = sum(combos.values())
    vio  = sum(v for k, v in combos.items() if k[0])
    ruta = sum(v for k, v in combos.items() if k[1])
    ok, falla = combos[(True, True, True)], combos[(True, True, False)]
    fallo_final[etiqueta] = 100 * falla / (ok + falla)
    print(f"\n{etiqueta} (n={n})")
    print(f"  perception names the class: {100*vio/n:>3.0f}%")
    print(f"  routing hits the class:     {100*ruta/n:>3.0f}%")
    print(f"  both correct:               {ok+falla}")
    print(f"  ... of those, action wrong: {falla} ({fallo_final[etiqueta]:.0f}%)")
    print(f"  full chain correct:         {ok} ({100*ok/n:.0f}%)")

### Perception levels

The system names the annotated object and marks it as safety-relevant, but rarely attributes
direct influence on the trajectory. That the ceiling holds under oracle localisation rules out
an identification deficit.

In [ ]:
niveles_datos = {}
for campo, etiqueta in [("A_con_skg", "1 frame"), ("temporal_10f", "10 frames"),
                        ("oraculo_bbox", "bounding box")]:
    n = m = rel = dir_ = 0
    for r in datos:
        f = r.get(campo)
        p = f.get("percepcion") if f else None
        if not p:
            continue
        n += 1
        objs = p.get("objects", [])
        nombra = lambda sel: menciona(" ".join(str(o.get("label", "")) for o in sel), r["tipo"])
        m += nombra(objs)
        rel += nombra([o for o in objs if o.get("relevant_to_safety")])
        dir_ += nombra([o for o in objs if o.get("influence_on_ego_path") == "direct"])
    niveles_datos[etiqueta] = [100*m/n, 100*rel/n, 100*dir_/n]
    print(f"{etiqueta:<16} names {100*m/n:>3.0f}% | safety-relevant {100*rel/n:>3.0f}% | "
          f"direct influence {100*dir_/n:>3.0f}%")

### Node-level precision

Only over the correctly routed scenarios, so the figure reflects how the knowledge resolves
rather than how well the router chose. Nodes whose condition rests on a directly observable
attribute do far better than those demanding a spatial judgement.

In [ ]:
por_nodo = {}
for r in datos:
    e = etapas(r)
    h = r["A_con_skg"].get("hazard")
    if e and h and e[1]:
        d = por_nodo.setdefault(h, {"n": 0, "ok": 0})
        d["n"] += 1
        d["ok"] += e[2]

print(f"{'node':<30}{'hits':>10}{'%':>7}")
for h, d in sorted(por_nodo.items(), key=lambda x: -x[1]["ok"] / x[1]["n"]):
    print(f"{h:<30}{d['ok']:>5}/{d['n']:<4}{100*d['ok']/d['n']:>6.0f}%")

print(f"\nTotal correctly routed: {sum(d['n'] for d in por_nodo.values())}")
seleccionados = set(Counter(r["A_con_skg"].get("hazard") for r in datos))
print(f"Nodes never selected: {sorted(set(CLASE_DE) - seleccionados)}")

### Figure — perception funnel and final failure rate

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2),
                               gridspec_kw={"width_ratios": [1.5, 1]})

etiquetas_nivel = ["Names\nthe class", "Marks as\nrelevant", "Direct\ninfluence"]
x = np.arange(3); ancho = 0.26
for i, (etiqueta, color) in enumerate(zip(niveles_datos, [COLOR_A, COLOR_B, COLOR_ALT])):
    ax1.bar(x + (i - 1) * ancho, niveles_datos[etiqueta], ancho,
            label=etiqueta, color=color)
ax1.set_xticks(x); ax1.set_xticklabels(etiquetas_nivel, fontsize=9)
ax1.set_ylabel("Scenarios (%)"); ax1.set_ylim(0, 105)
ax1.legend(frameon=False, fontsize=8.5)
ax1.set_title("Perception funnel", fontsize=10)

barras = ax2.bar(list(fallo_final), list(fallo_final.values()), 0.5,
                 color=[COLOR_A, COLOR_B, COLOR_ALT])
for bar, v in zip(barras, fallo_final.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 1.5, f"{v:.0f}%",
             ha="center", fontsize=9)
ax2.set_ylabel("Failure rate (%)"); ax2.set_ylim(0, 100)
ax2.set_title("Action errors when perception\nand routing both succeed", fontsize=10)
ax2.tick_params(axis="x", labelsize=9)

plt.tight_layout()
guardar(fig, "fig_embudo_percepcion")
plt.show()

## 3.4 — Captioning and validation

The caption is compared against the DRAMA reference with both lexical overlap and semantic
similarity metrics. Having both families is what separates stylistic divergence from incorrect
content.

In [ ]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

suavizado = SmoothingFunction().method1
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# One of the caption prompt examples ended up inside the evaluation subset when it
# was extended, so that scenario is excluded from the metrics.
CONTAMINADO = {10789}


def evaluar_captions(campo, etiqueta):
    pares = [(r[campo] if isinstance(r[campo], str) else r[campo].get("caption", ""),
              r["caption_gt"])
             for r in datos
             if r.get(campo) and r.get("caption_gt") and r["indice"] not in CONTAMINADO]
    pares = [(h, ref) for h, ref in pares if h and ref]

    bleu = meteor = rouge = 0.0
    for hip, ref in pares:
        bleu += sentence_bleu([ref.split()], hip.split(), weights=(0.25,)*4,
                              smoothing_function=suavizado)
        meteor += meteor_score([ref.split()], hip.split())
        rouge += scorer.score(ref, hip)["rougeL"].fmeasure
    n = len(pares)
    longitud = sum(len(h.split()) for h, _ in pares) / n
    empieza = sum(1 for h, _ in pares if h.lower().startswith("there is"))
    print(f"{etiqueta:<22} n={n} | BLEU-4 {bleu/n:.4f} | METEOR {meteor/n:.4f} | "
          f"ROUGE-L {rouge/n:.4f}")
    print(f"{'':22} length {longitud:.1f} | starts with 'There is' {100*empieza/n:.0f}%")
    return pares


ref_len = (sum(len(r["caption_gt"].split()) for r in datos if r.get("caption_gt"))
           / sum(1 for r in datos if r.get("caption_gt")))
print(f"Reference caption length: {ref_len:.1f} words\n")

# The few-shot caption lives under a different key in each condition
for r in datos:
    r["_capA"] = r["A_con_skg"].get("caption_fewshot", "")
pares_A = evaluar_captions("_capA", "A (with SKG)")
pares_B = evaluar_captions("B_caption", "B (no knowledge)")

In [ ]:
from bert_score import score as bertscore

for pares, etiqueta in [(pares_A, "A"), (pares_B, "B")]:
    _, _, f1 = bertscore([h for h, _ in pares], [r for _, r in pares],
                         lang="en", device="cpu", verbose=False)
    print(f"BERTScore F1 {etiqueta}: {f1.mean().item():.4f}")

### Figure — caption metrics, zero-shot against few-shot

In [ ]:
# Zero-shot values come from the earlier caption configuration, reported in Section 3.4
metricas = {
    "BLEU-4":    (0.033, 0.118),
    "METEOR":    (0.177, 0.345),
    "ROUGE-L":   (0.263, 0.375),
    "BERTScore": (0.878, 0.893),
}

fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(len(metricas)); ancho = 0.36
zs = [v[0] for v in metricas.values()]
fs = [v[1] for v in metricas.values()]

b1 = ax.bar(x - ancho/2, zs, ancho, label="Zero-shot", color=COLOR_B)
b2 = ax.bar(x + ancho/2, fs, ancho, label="Few-shot", color=COLOR_A)
for barras, vals in [(b1, zs), (b2, fs)]:
    for bar, v in zip(barras, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f"{v:.3f}",
                ha="center", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(list(metricas))
ax.set_ylabel("Score"); ax.set_ylim(0, 1.0)
ax.legend(frameon=False)
plt.tight_layout()
guardar(fig, "fig_metricas_caption")
plt.show()

### Validation and image verification

The validator checks the explanation against the perception and the retrieved knowledge. The
image verification is a separate check, done with the pipeline's own model, so it is
self-evaluation and its figures are an upper bound.

In [ ]:
val = [r["validacion"] for r in datos if r.get("validacion")]
n = len(val)
cuenta = lambda cond: sum(1 for v in val if cond(v))

print(f"--- Validator (n={n}) ---")
for etiqueta, cond in [
        ("Passed", lambda v: v.get("verdict") == "pass"),
        ("With hallucination", lambda v: v.get("hallucinated_claims")),
        ("Inconsistencies", lambda v: v.get("inconsistencies")),
        ("Ego-frame drift", lambda v: str(v.get("ego_frame_ok")).lower() == "false")]:
    k = cuenta(cond)
    print(f"  {etiqueta:<20} {k:>3} ({100*k/n:.0f}%)")

juicios = [r["juicio_automatico"] for r in datos if r.get("juicio_automatico")]
print(f"\n--- Image verification (n={len(juicios)}) ---")
for crit in ["grounding", "causal_validity", "action_support"]:
    v = [j[crit] for j in juicios if crit in j]
    print(f"  {crit:<18} {sum(v)/len(v):.2f}/3  {dict(sorted(Counter(v).items()))}")

por_clase = defaultdict(list)
for r in datos:
    if r.get("juicio_automatico"):
        por_clase[r["tipo"]].append(sum(r["juicio_automatico"].values()) / 3)
print()
for t in TIPOS:
    v = por_clase[t]
    print(f"  {t:<16} {sum(v)/len(v):.2f}/3 (n={len(v)})")

## Qualitative examples

Outputs with an explicit normative reference, one per agent class, over scenarios the system
resolved correctly. These go in the appendix.

In [ ]:
from PIL import Image
from matplotlib.gridspec import GridSpec
import textwrap

from datos import cargar_escenarios

por_ruta = {e["indice"]: e["ruta"] for e in cargar_escenarios()}
por_indice = {r["indice"]: r for r in datos}


def figura_ejemplo(indice, salida, accion, nombre):
    """Frame on the left, system output on the right."""
    r = por_indice[indice]
    gt_accion = normalizar_accion(r["sugerencia_gt"])[0] or r["sugerencia_gt"]

    fig = plt.figure(figsize=(12, 4.6))
    gs = GridSpec(1, 2, width_ratios=[1.35, 1], wspace=0.06)

    ax = fig.add_subplot(gs[0])
    ax.imshow(Image.open(por_ruta[indice]))
    ax.axis("off")
    ax.set_title(f"{r['tipo']} — scenario {indice}", fontsize=10, loc="left")

    ax2 = fig.add_subplot(gs[1])
    ax2.axis("off")
    y = 0.97

    def bloque(etiqueta, texto, negrita=False, ancho=52):
        nonlocal y
        ax2.text(0, y, etiqueta, fontsize=8.5, weight="bold", color=COLOR_A,
                 va="top", transform=ax2.transAxes)
        y -= 0.052
        for linea in textwrap.wrap(str(texto), ancho):
            ax2.text(0, y, linea, fontsize=8.5, va="top",
                     weight="bold" if negrita else "normal", transform=ax2.transAxes)
            y -= 0.048
        y -= 0.055

    bloque("DRAMA reference caption", r["caption_gt"])
    bloque("DRAMA suggested action", gt_accion, negrita=True)
    bloque("System output", salida)
    bloque("Recommended action", accion, negrita=True)

    guardar(fig, nombre)
    plt.show()

In [ ]:
# Generated with the normative-reference prompt, one per agent class
ejemplos = {
    5132: ("The ego-vehicle must stop at the red traffic light because it directly affects "
           "the ego direction and poses a safety risk if the vehicle continues without "
           "stopping. The Vienna Convention Art. 5.1 and 5.2 require compliance with light "
           "signals, where signals take precedence.", "(must) Stop"),
    979:  ("The ego-vehicle must manoeuvre carefully because a cyclist is riding in the same "
           "direction and is located mid-road, posing a potential collision risk. Vienna "
           "Convention Art. 11.4 states: 'Sufficient lateral space shall be maintained when "
           "overtaking.'", "Carefully manoeuvre"),
    802:  ("The ego-vehicle must stop because the white van directly ahead is fully stopped, "
           "increasing the risk of collision. This decision is supported by Vienna Convention "
           "Art. 13.5, which states that sufficient distance should be maintained to avoid "
           "collision if the vehicle ahead stops.", "(must) Stop"),
    8637: ("The ego-vehicle must stop because a pedestrian is crossing the street, which could "
           "pose a risk if they step into the path of the ego-vehicle. The pedestrian is "
           "located in the crosswalk. Vienna Convention Art. 21.2 (behaviour at pedestrian "
           "crossings).", "(must) Stop"),
}

for indice, (texto, accion) in ejemplos.items():
    figura_ejemplo(indice, texto, accion, f"ejemplo_{indice}")

For the vehicle example, the annotation describes a compact hatchback while the system reports
a van. The action is right despite the imprecise identification, which illustrates that the
manoeuvre can be reached by a route that does not quite correspond to the scene.